# Refining the writing guide with a dependency parse

**Work unit `2026-08-16_01_register-from-four-sources`.** This notebook reproduces the
analysis that changed `authoring/WRITING_GUIDE.md` from a sentence-statistics specification
into a discourse specification.

The question it answers: *the corpus passes every register threshold, so why does the prose
still not read like something a subject matter expert wrote?*

The method in one line: **parse the human sources and the generated corpus, find where they
diverge, then mine the sources for real sentences that show the right shape.** The parse
locates evidence. The sentences are the guide.

## How to run it

spaCy is deliberately **not** a project dependency. Nothing in the build needs it, and the
findings here are diagnostic rather than gates. Run the notebook with a throwaway environment:

```bash
uv run --with spacy \
  --with 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  --with jupyter jupyter lab
```

Everything else comes from the repository: prose extraction is reused from
`authoring/check_style.py` so this notebook and the register gate always read the same text.


## 0. Setup

Find the repository root, import the project's own prose extractors, and load the parser.


In [1]:
import re, sys, math, statistics as stat
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().resolve()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'authoring'))

# The project's own extractors. Reused rather than reimplemented, so the notebook measures
# exactly the text the register gate measures: no YAML, code chunks, tables or captions.
from check_style import (prose_from_qmd, prose_from_extract, sentences,
                         measure, LIMITS, _band)

import spacy
nlp = spacy.load('en_core_web_sm')
print('repo root:', ROOT)
print('spaCy', spacy.__version__)

repo root: /home/moritz/github_repos/synthetic_data
spaCy 3.8.15


## 1. The two corpora

**Reference**: the published human documents the corpus is built on, as page-marked text
extracts in `refs/text/`. The page ranges are the running-prose chapters; front matter and
appendices are excluded because they are lists, not prose.

**Target**: the generated documents. Inline `{python}` expressions are replaced by a numeral
so the sentence keeps the shape a reader sees — a sentence whose subject is a computed value
still has a subject.


In [2]:
REF_SOURCES = [('A-Mab', 'refs/text/amab.txt', 60, 200),
               ('PDA TR60', 'refs/text/pda60.txt', 18, 90)]
TARGET_DOCS = ['PCR-003_bioreactor', 'PCR-005_protein_a',
               'PCR-008_aex', 'PCP-003_bioreactor']

def load_reference(path, lo, hi):
    return sentences(prose_from_extract(str(ROOT / path), lo, hi))

def load_target(stem):
    text = prose_from_qmd(str(ROOT / 'pc_package' / f'{stem}.qmd'))
    return [s.replace('NUM', '12.3') for s in sentences(text)]

REF = {name: load_reference(p, lo, hi) for name, p, lo, hi in REF_SOURCES}
TGT = {stem.split('_')[0]: load_target(stem) for stem in TARGET_DOCS}

for name, ss in {**REF, **TGT}.items():
    print(f'{name:<10s} {len(ss):5d} sentences')

A-Mab       1205 sentences
PDA TR60     997 sentences
PCR-003      423 sentences
PCR-005      374 sentences
PCR-008      414 sentences
PCP-003      226 sentences


### The trap that comes first: boilerplate

`prose_from_extract` carries hand-written filters for each source — `Licensed to`,
`Technical Report No`, `CMC Biotech Working Group`. Skip that step and the measurement is
garbage rather than merely noisy.

The two ISPE guides in `$SYNTHETIC_DATA_SOURCES` carry a per-page DRM footer. Measured
without a filter, 37.1 % of their sentences came out under 15 words against a human band of
15–32 %, and **300 of the 470 short sentences were the same four watermark lines repeated on
every page**. With the footer removed the same document sits inside the band.

The cell below reproduces that, and skips cleanly when the sources are not on this machine.
They live outside the repository and are not redistributable.


In [3]:
import os
SRC_DIR = Path(os.environ.get('SYNTHETIC_DATA_SOURCES',
    '/home/moritz/Nextcloud/Datasets/synthetic_data/source_documents'))
ISPE_BOILER = ('Downloaded from', 'For personal use only', 'No other uses',
               'For individual use only', 'Copyright ISPE', 'guidance-docs.ispe.org')

def show_boilerplate_effect(pdf_name, pages=(30, 190)):
    path = SRC_DIR / pdf_name
    if not path.exists():
        print(f'skip: {pdf_name} not present'); return
    try:
        import fitz
    except ImportError:
        print('skip: PyMuPDF not installed'); return
    doc = fitz.open(path)
    raw = [p.get_text('text') for p in doc][pages[0]:pages[1]]
    doc.close()
    lines = [l.strip() for page in raw for l in page.splitlines() if len(l.strip()) >= 30]
    kept = [l for l in lines if not any(b in l for b in ISPE_BOILER)]
    dropped = len(lines) - len(kept)
    for label, block in (('with boilerplate', lines), ('filtered', kept)):
        ss = sentences(' '.join(block))
        short = 100 * sum(1 for s in ss if len(s.split()) < 15) / max(len(ss), 1)
        print(f'  {label:<18s} {len(ss):5d} sentences, {short:5.1f} % under 15 words')
    print(f'  boilerplate lines dropped: {dropped}')

show_boilerplate_effect('2023-ispe-good-practice-guide-technology-transfer-(third-edition).pdf',
                        pages=(30, 140))

  with boilerplate    1635 sentences,  41.2 % under 15 words
  filtered            1195 sentences,  19.5 % under 15 words
  boilerplate lines dropped: 330


## 2. Why the existing gate could not find this

`check_style.py` enforces thirteen thresholds, every one read off the human sources. Run it
over the corpus and everything passes. That is the whole problem: the prose is wrong and the
instrument says it is right.


In [4]:
rows = {}
for name, ss in {**REF, **TGT}.items():
    rows[name] = measure(' '.join(ss))[0]

keys = ['mean_len', 'pct_over_40', 'pct_under_15', 'em_dash', 'semicolon', 'colon', 'paren']
hdr = f"{'metric':<34s}{'band':>10s}" + ''.join(f'{n:>11s}' for n in rows)
print(hdr); print('-' * len(hdr))
for k in keys:
    lo, hi, desc = LIMITS[k]
    print(f'{desc:<34s}{_band(lo, hi):>10s}' + ''.join(f'{rows[n][k]:>11.2f}' for n in rows))
print()
print('Every corpus document is inside every band. The gate is not lying; it is measuring')
print('sentence geometry and punctuation, and the defect is not there.')

metric                                  band      A-Mab   PDA TR60    PCR-003    PCR-005    PCR-008    PCP-003
--------------------------------------------------------------------------------------------------------------
mean sentence length (words)       20.0-30.5      26.61      23.63      23.39      23.45      23.48      24.61
% of sentences over 40 words        3.0-21.5      13.61       8.93       5.60       4.86       9.93       7.17
% of sentences under 15 words      15.0-32.0      20.17      22.57      21.65      27.03      26.39      18.83
em-dashes per 1k words                 <=2.5       0.00       1.02       0.00       0.00       0.00       0.00
semicolons per 1k words                <=4.5       1.00       1.99       0.00       0.69       0.00       0.00
colons per 1k words                    <=5.5       3.15       2.59       0.62       0.46       0.62       2.19
parenthetical openings per 1k words  3.0-14.5      11.48      11.76      12.07       8.41       9.18       8.74


## 3. Look at the sentence a reader complained about

Two sentences making the **same argument** — a statistically significant effect that does not
matter in practice. One is easy to read. One is not. The parse says why.


In [5]:
EASY = ('In this case, changing the medium concentration from 0.8 to 1.6 X '
        'only changed the aFucosylation levels by 0.3 %.')
HARD = ('These are large and well-resolved effects of limited practical consequence, '
        'because the attribute is of very low criticality and its acceptance criterion '
        'is applied as an upper limit that lies far above the observed range.')

def anatomy(text, label):
    d = nlp(text)
    root = next(t for t in d if t.dep_ == 'ROOT')
    subs = [t.text for t in d if t.dep_ in ('nsubj', 'nsubjpass')]
    absn = [t.text for t in d if t.pos_ == 'NOUN'
            and re.search(r'(tion|ment|ence|ance|ity|ness|quence)$', t.text, re.I)]
    preps = [t.text for t in d if t.dep_ == 'prep']
    print(f'-- {label} --')
    print(f'   ROOT            {root.text!r}  ({root.pos_}, lemma {root.lemma_!r})')
    print(f'   subjects        {subs}')
    print(f'   abstract nouns  {absn}')
    print(f'   prepositions    {preps}')
    print(f'   clauses         {sum(1 for t in d if t.dep_ in ("ROOT","ccomp","advcl","relcl","conj"))}')
    print()

anatomy(EASY, 'A-Mab')
anatomy(HARD, 'PCR-003')

-- A-Mab --
   ROOT            'changed'  (VERB, lemma 'change')
   subjects        []
   abstract nouns  ['concentration']
   prepositions    ['In', 'from', 'by']
   clauses         1

-- PCR-003 --
   ROOT            'are'  (AUX, lemma 'be')
   subjects        ['These', 'attribute', 'criterion', 'that']
   abstract nouns  ['consequence', 'criticality', 'acceptance']
   prepositions    ['of', 'of', 'as', 'above']
   clauses         5



The A-Mab sentence has a lexical event verb (`changed`) and one abstract noun.

The corpus sentence has **`are`** for a root, a bare demonstrative subject whose antecedent is
a table, a noun for a predicate, three nominalisations and four stacked prepositions.
**Nothing happens in it.** The reader has to reconstruct the events from the nouns.

Render the trees to see it — this is the view at <https://spacy.io/usage/visualizers/>.


In [6]:
from spacy import displacy

def draw(text):
    doc = nlp(text)
    try:
        displacy.render(doc, style='dep', jupyter=True,
                        options={'compact': True, 'distance': 95, 'word_spacing': 25})
    except Exception:
        for t in doc:
            print(f'{t.text:<18s} {t.dep_:<10s} -> {t.head.text}')

draw(EASY)

In [7]:
draw(HARD)

## 4. Turn the observation into a measurement

One passage proves nothing. Define per-sentence features and run them over both corpora.
Every feature is one an author could act on.


In [8]:
def depth(tok):
    return 1 + max((depth(k) for k in tok.children), default=0)

def prep_chain(tok):
    best = 0
    for k in tok.children:
        if k.dep_ in ('prep', 'agent'):
            inner = max((prep_chain(p) for p in k.children if p.dep_ == 'pobj'), default=0)
            best = max(best, 1 + inner)
    return best

def sentence_features(doc):
    root = next((t for t in doc if t.dep_ == 'ROOT'), None)
    if root is None:
        return None
    subj = next((t for t in doc if t.dep_ in ('nsubj', 'nsubjpass')), None)
    return {
        'depth': depth(root),
        'prep_chain': max((prep_chain(t) for t in doc), default=0),
        'copula': int(root.lemma_ == 'be'),
        'preverb': root.i if root.pos_ in ('VERB', 'AUX') else 0,
        'finite': sum(1 for t in doc if t.pos_ in ('VERB', 'AUX')
                      and t.dep_ in ('ROOT', 'ccomp', 'advcl', 'relcl', 'conj', 'xcomp')),
        'relcl': sum(1 for t in doc if t.dep_ == 'relcl'),
    }

def profile(sents, limit=450):
    rows = [f for f in (sentence_features(nlp(s)) for s in sents[:limit]) if f]
    out = {k: stat.mean(r[k] for r in rows) for k in rows[0] if k != 'copula'}
    out['copula%'] = 100 * stat.mean(r['copula'] for r in rows)
    out['n'] = len(rows)
    return out

PROF = {name: profile(ss) for name, ss in {**REF, **TGT}.items()}
cols = ['depth', 'prep_chain', 'finite', 'relcl', 'preverb', 'copula%', 'n']
hdr = f"{'feature':<14s}" + ''.join(f'{n:>11s}' for n in PROF)
print(hdr); print('-' * len(hdr))
for c in cols:
    print(f'{c:<14s}' + ''.join(f'{PROF[n][c]:>11.2f}' for n in PROF))

feature             A-Mab   PDA TR60    PCR-003    PCR-005    PCR-008    PCP-003
--------------------------------------------------------------------------------
depth                7.75       7.37       6.65       6.91       6.98       6.82
prep_chain           1.37       1.34       1.22       1.37       1.33       1.25
finite               2.43       2.38       2.29       2.25       2.27       2.63
relcl                0.27       0.24       0.27       0.27       0.28       0.35
preverb              8.68       9.06       5.83       6.88       5.71       6.31
copula%             14.67      18.22      32.39      26.74      22.71      27.43
n                  450.00     450.00     423.00     374.00     414.00     226.00


Read the table twice.

**Tree depth, prepositional nesting, finite verbs and relative clauses are the same or
*lower* in the corpus.** The generated prose is not more complex than the human prose. Any
rule about shortening or simplifying sentences would be aimed at something that is not there.

**`copula%` and `preverb` diverge.** A third of PCR-003 sentences have `be` for a main verb
against roughly one in six of the sources, and the corpus puts fewer tokens in front of the
verb.

Selecting features by hand is a weakness, though: I chose them after reading two sentences.
The next section removes that judgement.


## 5. Let the data pick the features

Emit a bag of structural features per sentence — dependency labels and
`head-POS < relation < child-POS` triples — then rank by log ratio between the two corpora.
No hypothesis about what matters.


In [9]:
def bag(doc):
    out = []
    root = next((t for t in doc if t.dep_ == 'ROOT'), None)
    if root is not None:
        out.append(f'ROOT_POS={root.pos_}')
        out.append(f'FRONT_LEN={min(root.i // 3, 4)}')
    if len(doc) > 1:
        out.append(f'OPEN_POS={doc[0].pos_}')
    for t in doc:
        out.append(f'dep={t.dep_}')
        if t.head.i != t.i:
            out.append(f'tri={t.head.pos_}<{t.dep_}<{t.pos_}')
    return out

def bag_profile(sents, limit=900):
    c, n = Counter(), 0
    for s in sents[:limit]:
        c.update(bag(nlp(s))); n += 1
    return c, n

ref_all = [s for ss in REF.values() for s in ss]
tgt_all = [s for ss in TGT.values() for s in ss]
rc, rn = bag_profile(ref_all)
tc, tn = bag_profile(tgt_all)

MIN = 25
ranked = []
for k in set(rc) | set(tc):
    if rc[k] + tc[k] < MIN:
        continue
    rr, tr = (rc[k] + .5) / rn, (tc[k] + .5) / tn
    ranked.append((math.log2(tr / rr), k, rc[k] / rn, tc[k] / tn))
ranked.sort()

print(f'reference {rn} sentences   target {tn} sentences   (rates per sentence)\n')
print('the corpus does this MUCH LESS')
for lr, k, r, t in ranked[:8]:
    print(f'  {lr:6.2f}  {k:<28s}{r:8.3f}{t:8.3f}')
print('\nthe corpus does this MUCH MORE')
for lr, k, r, t in ranked[-8:][::-1]:
    print(f'  {lr:6.2f}  {k:<28s}{r:8.3f}{t:8.3f}')

reference 900 sentences   target 900 sentences   (rates per sentence)

the corpus does this MUCH LESS
   -6.23  tri=NOUN<punct<SYM             0.041   0.000
   -5.18  tri=PROPN<nmod<PROPN           0.060   0.001
   -4.84  tri=VERB<prep<VERB             0.079   0.002
   -3.70  OPEN_POS=ADV                   0.079   0.006
   -3.13  dep=expl                       0.043   0.004
   -3.13  tri=VERB<expl<PRON             0.043   0.004
   -2.60  tri=PROPN<amod<ADJ             0.057   0.009
   -2.45  tri=NOUN<compound<PROPN        0.356   0.064

the corpus does this MUCH MORE
    3.80  tri=VERB<dobj<PRON             0.004   0.069
    3.37  tri=ADP<conj<ADP               0.004   0.051
    3.23  tri=AUX<attr<NUM               0.002   0.026
    3.09  tri=ADP<cc<CCONJ               0.007   0.061
    3.02  tri=NUM<prep<ADP               0.006   0.049
    2.78  tri=ADP<pobj<PRON              0.017   0.118
    2.65  tri=NUM<det<DET                0.006   0.038
    2.27  tri=AUX<cc<CCONJ               

Three things come out of this ranking, and one of them is a warning.

1. **`dep=aux` collapses** — roughly 1.0 auxiliaries per sentence in the sources against 0.2
   in the corpus. Auxiliaries are where modality lives: *can*, *may*, *should*, *will*,
   *has been*.
2. **`OPEN_POS=ADV` is about five times rarer.** Sentences that open with an adverb are the
   ones that open with a connective — *However*, *Also*, *Therefore*, *Similarly*.
3. **`dep=poss` and `NOUN<poss<PRON` are five times more common in the corpus.** Nobody
   predicted this one; reading the passages had not surfaced it.

**The warning:** several of the top rows are artifacts of replacing `{python}` expressions
with a numeral (`OPEN_POS=NUM`, `ADP<pobj<NUM`). The ranking finds where two corpora differ.
Deciding which differences are style and which are measurement is still a person's job.


## 6. Chase the finding nobody predicted

Possessives. Check the raw rate before believing the parse.


In [10]:
def per_1k(text, pattern):
    return 1000 * len(re.findall(pattern, text, re.I)) / max(len(text.split()), 1)

ref_text = ' '.join(ref_all)
print(f"{'':<12s}{'its':>8s}{'their':>8s}{'it':>8s}")
print(f"{'A-Mab+PDA':<12s}" + ''.join(f'{per_1k(ref_text, p):>8.2f}'
      for p in (r'\bits\b', r'\btheir\b', r'\bit\b')))
for name, ss in TGT.items():
    t = ' '.join(ss)
    print(f'{name:<12s}' + ''.join(f'{per_1k(t, p):>8.2f}'
          for p in (r'\bits\b', r'\btheir\b', r'\bit\b')))

print('\nWhat it looks like in the corpus:')
for m in list(re.finditer(r'[^.]*\bits\b[^.]*\.', ' '.join(TGT['PCR-003'])))[:5]:
    s = m.group(0).strip()
    if 10 < len(s.split()) < 35:
        print('  *', s[:130])

                 its   their      it
A-Mab+PDA       0.36    0.70    2.18
PCR-003         0.31    0.83    8.22
PCR-005         4.61    2.07    9.56
PCR-008         3.92    2.27    7.01
PCP-003         0.36    0.18   12.02

What it looks like in the corpus:
  * That finding needs its context, and the context does not remove it.
  * It does not set the aggregate level of the drug substance on its own, because cation exchange is the principal aggregate polishing
  * The instrument was inside its calibration interval when the run was executed, with the next calibration due on 12.


`its` runs about twenty times the A-Mab rate. Every one of them makes the reader hold an
antecedent and bind it: *its acceptance criterion*, *its characterized range*, *its
set-point*, *its limit*. A-Mab writes *the* acceptance criterion, or names the thing.

The sentence the project owner flagged does it twice.


## 7. Test the obvious explanations, and discard them

A rule that fixes nothing is worse than no rule, because it costs an author attention. Each
hypothesis below sounds right and is false.


In [11]:
def svo_audit(sents, limit=900):
    n = before = obj_after = obj_before = 0
    for s in sents[:limit]:
        d = nlp(s)
        root = next((t for t in d if t.dep_ == 'ROOT'), None)
        if root is None or root.pos_ not in ('VERB', 'AUX'):
            continue
        # NB: token.head returns a fresh object, so compare by index, never with `is`.
        subj = next((t for t in d if t.dep_ in ('nsubj', 'nsubjpass')
                     and t.head.i == root.i), None)
        if subj is None:
            continue
        n += 1
        before += subj.i < root.i
        obj = next((t for t in d if t.dep_ in ('dobj', 'attr', 'acomp')
                    and t.head.i == root.i), None)
        if obj is not None:
            obj_after += obj.i > root.i
            obj_before += obj.i < root.i
    return dict(n=n, subj_first=100 * before / n,
                obj_last=100 * obj_after / max(obj_after + obj_before, 1))

print('Hypothesis: the corpus inverts word order.')
for name, ss in {**REF, **TGT}.items():
    a = svo_audit(ss)
    print(f"  {name:<10s} subject before verb {a['subj_first']:6.1f} %   "
          f"object after verb {a['obj_last']:6.1f} %   ({a['n']} clauses)")
print('\nVerdict: FALSE. Canonical order is near absolute on both sides.')

Hypothesis: the corpus inverts word order.


  A-Mab      subject before verb  100.0 %   object after verb  100.0 %   (743 clauses)


  PDA TR60   subject before verb   99.9 %   object after verb  100.0 %   (784 clauses)


  PCR-003    subject before verb  100.0 %   object after verb  100.0 %   (399 clauses)


  PCR-005    subject before verb  100.0 %   object after verb   99.4 %   (350 clauses)


  PCR-008    subject before verb  100.0 %   object after verb  100.0 %   (398 clauses)


  PCP-003    subject before verb  100.0 %   object after verb  100.0 %   (216 clauses)

Verdict: FALSE. Canonical order is near absolute on both sides.


In [12]:
NOMINAL = re.compile(r'(?:tion|sion|ment|ance|ence|ity|ness|ency|ation)$', re.I)

def null_tests(sents, limit=600):
    docs = [nlp(s) for s in sents[:limit]]
    words = sum(len([t for t in d if not t.is_punct]) for d in docs) or 1
    nomof = sum(1 for d in docs for t in d if t.pos_ == 'NOUN' and NOMINAL.search(t.text)
                and any(c.dep_ == 'prep' and c.lower_ == 'of' for c in t.children))
    toks = [t.lower_ for d in docs for t in d if not t.is_punct and t.lower_ != '12.3']
    grams = Counter(tuple(toks[i:i+4]) for i in range(len(toks) - 3))
    formulaic = sum(c for c in grams.values() if c >= 3)
    def conj_len(t):
        return max((1 + conj_len(k) for k in t.children if k.dep_ == 'conj'), default=0)
    return dict(
                nomof=1000 * nomof / words,
                formulaic=1000 * formulaic / words,
                conj=stat.mean(max((conj_len(t) for t in d), default=0) for d in docs))

print(f"{'':<12s}{'nominalisation+of':>20s}{'repeated 4-grams':>19s}{'coordination':>14s}")
for name, ss in {**REF, **TGT}.items():
    r = null_tests(ss)
    print(f"{name:<12s}{r['nomof']:>20.2f}{r['formulaic']:>19.2f}{r['conj']:>14.2f}")
print('\nVerdict: FALSE on all three. The corpus nominalises HALF as often as the sources,')
print('repeats 4-grams at a rate comparable to A-Mab (41-62 vs 59), and coordinates')
print('identically. None of the three supports a rule.')

               nominalisation+of   repeated 4-grams  coordination


A-Mab                       8.90              59.28          0.94


PDA TR60                   10.97              26.85          0.88


PCR-003                     2.98              42.61          0.82


PCR-005                     2.93              61.86          0.75


PCR-008                     5.25              49.11          0.72


PCP-003                     1.79              22.41          0.86

Verdict: FALSE on all three. The corpus nominalises HALF as often as the sources,
repeats 4-grams at a rate comparable to A-Mab (41-62 vs 59), and coordinates
identically. None of the three supports a rule.


Eight hypotheses were tested this way in total. All eight returned null: sentence length,
structural complexity, word order, nominalisation, formulaic repetition, over-claiming,
coordination and number density.

**That is what bounds the work.** It means no threshold of the kind `check_style.py` already
carries can reach the defect, and it stops six plausible rules from being written into the
guide.


## 8. The defect is in how sentences connect

Two discourse measurements. The first is a plain count and needs no parser. The second needs
the parse, and it is the one that matters most.


In [13]:
CONNECTIVES = ['However', 'For example', 'By contrast', 'In addition',
               'As a result', 'Note that', 'Therefore', 'Consequently']
print(f"{'':<14s}" + ''.join(f'{c[:11]:>13s}' for c in CONNECTIVES))
for name, path, lo, hi in REF_SOURCES:
    raw = (ROOT / path).read_text(errors='ignore')
    print(f'{name:<14s}' + ''.join(
        f'{len(re.findall(c, raw, re.I)):>13d}' for c in CONNECTIVES))
for stem in TARGET_DOCS:
    raw = (ROOT / 'pc_package' / f'{stem}.qmd').read_text()
    print(f"{stem.split('_')[0]:<14s}" + ''.join(
        f'{len(re.findall(c, raw, re.I)):>13d}' for c in CONNECTIVES))

                    However  For example  By contrast  In addition  As a result    Note that    Therefore  Consequentl
A-Mab                    46           12            3           23            4           11           69            3
PDA TR60                 21           22            1           21            2            3            6            1
PCR-003                   2            0            1            0            0            0            9            1
PCR-005                   0            0            0            0            1            0            5            0
PCR-008                   0            0            0            0            0            0            7            0
PCP-003                   2            0            1            0            1            0            3            0


Zero `However` and zero `For example` across four documents and roughly 30,000 words. The
whole repertoire has collapsed onto `therefore` — which `check_style.py` happens to **cap** at
1.2 per 1000 words, so the gate restricts the one connective still in service.

Now the parse-based measure: does a sentence start from what the previous one was about?


In [14]:
def topic_chaining(sents, limit=600):
    docs = [nlp(s) for s in sents[:limit]]
    chained = pairs = 0
    for prev, cur in zip(docs, docs[1:]):
        subj = next((t for t in cur if t.dep_ in ('nsubj', 'nsubjpass')), None)
        if subj is None:
            continue
        pairs += 1
        prev_l = {t.lemma_.lower() for t in prev if t.pos_ in ('NOUN', 'PROPN', 'ADJ')}
        subj_l = {t.lemma_.lower() for t in subj.subtree if t.pos_ in ('NOUN', 'PROPN', 'ADJ')}
        chained += bool(subj_l & prev_l) or subj.pos_ == 'PRON'
    return 100 * chained / max(pairs, 1)

for name, ss in {**REF, **TGT}.items():
    print(f'{name:<12s} {topic_chaining(ss):5.1f} % of sentences continue the previous topic')

A-Mab         59.0 % of sentences continue the previous topic


PDA TR60      59.4 % of sentences continue the previous topic


PCR-003       30.7 % of sentences continue the previous topic


PCR-005       34.8 % of sentences continue the previous topic


PCR-008       32.8 % of sentences continue the previous topic


PCP-003       34.4 % of sentences continue the previous topic


**This is the largest single finding.** Human sources chain roughly 58 % of sentences; the
corpus chains about a third, so two thirds of its sentences start a fresh topic and the reader
is re-oriented on nearly every one.

And `WRITING_GUIDE.md` §2d **already says this**: *"Begin with information the reader already
has and end with the new information."* The rule exists and is met a third of the time. That
makes it a rule to **exemplify and check**, not to invent — the cheapest repair available.


## 9. Why the authors wrote it this way

Two rules in the guide forbid the shapes that carry an argument:

> §2c "One paragraph, one point. Open with the point, then give the evidence."
> §2d "One sentence, one point; if a sentence carries two claims, make it two sentences."

| Move | What it needs | §2c/§2d verdict |
|---|---|---|
| `However` | a claim and its counter-consideration | two points — split them |
| `For example` | a rule and an instance | two points — split them |
| `By contrast` | two things compared | two points — split them |

**The authors complied exactly.** The defect is in the specification, not the execution.


## 10. Mine the sources for the shapes that should replace them

This is where the parse stops being a measuring instrument and becomes a search tool. For each
move we want an author to imitate, pull the real sentences that instantiate it.


In [15]:
def clean(s):
    if not (8 <= len(s.split()) <= 45):
        return False
    if any(x in s for x in ('=====', 'Page ', 'Licensed', 'Copyright', 'Table ', 'Figure ')):
        return False
    return sum(c.isdigit() for c in s) <= len(s) / 5

POOL = [(name, s) for name, ss in REF.items() for s in ss if clean(s)]
CHANGE = {'change', 'increase', 'decrease', 'reduce', 'raise', 'lower', 'strip', 'result'}

def mine(predicate, n=4):
    out = []
    for src, s in POOL:
        if predicate(nlp(s), s):
            out.append((src, s.strip()))
        if len(out) >= n:
            break
    return out

PATTERNS = {
    'the main verb names the event':
        lambda d, s: (r := next((t for t in d if t.dep_ == 'ROOT'), None)) is not None
                     and r.lemma_ in CHANGE,
    'concede, then commit':
        lambda d, s: re.match(r'\s*(However|Although|While)\b', s) is not None,
    'frame before the subject':
        lambda d, s: d[0].pos_ == 'ADP' and any(t.dep_ == 'punct' and t.text == ','
                                                for t in d[:8]),
    'modality carries the risk posture':
        lambda d, s: any(t.tag_ == 'MD' for t in d),
}

for title, pred in PATTERNS.items():
    print(f'### {title}')
    for src, s in mine(pred):
        print(f'  [{src}] {s[:165]}')
    print()

### the main verb names the event


  [A-Mab] Rationale for control strategy based on design space and risk assessment results Demonstration of how the design space is applicable to multiple operational scales a
  [A-Mab] The optimized process resulted in a higher integral of the viable cell concentration, longer culture duration and thus higher product titers.
  [A-Mab] The lower temperature also resulted in slightly higher levels of a-fucosylation and slightly lower galactosylation.
  [A-Mab] Lower pH also resulted in slightly increased levels of a-fucosylation.

### concede, then commit


  [A-Mab] However, since experience with other mAbs has shown that the N-1 seed bioreactor can potentially affect product quality, process characterization and seed-to-product
  [A-Mab] Although key process parameters and key process attributes have been shown not to impact product quality, they are included in the control strategy because their mon
  [A-Mab] However as part of continuous process monitoring in commercial operations, a PCA model will be developed for A-Mab once sufficient full scale data becomes available.
  [A-Mab] However, there are other scale-dependent parameters that must be considered for successful and consistent process performance when operating at various scales.

### frame before the subject


  [A-Mab] For this, the seed cultures are expanded through multiple passages by increasing the volume and/or number of disposable culture vessels in Step 1 and by increasing t
  [A-Mab] For the purposes of this case study, only a subset of quality attributes was considered in the analysis of drug substance and drug product development; these include
  [A-Mab] In a real-life case scenario, the examples and approaches described here would include all relevant product quality and material attributes.
  [A-Mab] In order to meet anticipated commercial demand, Process 1 was further optimized to increase product titers while ensuring no significant impact on product quality.

### modality carries the risk posture


  [A-Mab] Here, we recognize that traditional approaches can span the gamut from using One-Factor-At-a-Time (OFAT) experiments to full DOEs, and that many larger and well esta
  [A-Mab] To provide flexibility in the manufacturing schedule, the seed cultures can be maintained for additional culture passages or used to generate additional inoculum tra
  [A-Mab] Process 1 represents a well established platform with extensive process performance history and thus provided a high level of assurance that the desired quality attr
  [A-Mab] In a real-life case scenario, the examples and approaches described here would include all relevant product quality and material attributes.



These are the examples that go into `REGISTER_EXEMPLAR.md`. They are not invented, and they
are not paraphrased — which matters, because the exemplar file is gated.


## 11. Verify every mined quote before it reaches the guide

`authoring/check_exemplar_quotes.py` requires every blockquote in the exemplar to be verbatim
in `refs/text/`. This is not a formality: of 25 quotes mined for this work unit, **one failed**
— a sentence that spans a page break, which `prose_from_extract` reconstructs across the
running header while the raw file does not contain it contiguously.

A mined sentence can look verbatim and be an artifact of the extractor.


In [16]:
def verbatim(quote, source_key):
    raw = (ROOT / f'refs/text/{source_key}.txt').read_text(errors='ignore')
    return re.sub(r'\s+', ' ', quote) in re.sub(r'\s+', ' ', raw)

CHECKS = [
    ('amab', 'Longer culture times resulted in higher titers and lower a-fucosylation levels.'),
    ('amab', 'Although the extent of the effects may differ slightly, viral clearance decreases '
             'as pH decreases and conductivity increases.'),
    ('pda60', 'Since screening designs do not always clearly identify interactions, the reduced '
              'number of parameters identified by the screening experiment will be included in '
              'further experiments.'),
    ('amab', 'The drilled pipes produce larger bubbles and thus a lower volumetric mass transfer'),
]
for key, q in CHECKS:
    print(f"{'OK  ' if verbatim(q, key) else 'FAIL'}  [{key}] {q[:78]}...")
print('\nThe last one is the page-break case: real prose, not a contiguous substring.')

OK    [amab] Longer culture times resulted in higher titers and lower a-fucosylation levels...
OK    [amab] Although the extent of the effects may differ slightly, viral clearance decrea...
OK    [pda60] Since screening designs do not always clearly identify interactions, the reduc...
FAIL  [amab] The drilled pipes produce larger bubbles and thus a lower volumetric mass tran...

The last one is the page-break case: real prose, not a contiguous substring.


## 12. What changed in the guide

Each edit traces to a measurement above. Nothing was added on taste.

| Finding | Measurement | Change |
|---|---|---|
| argument shapes are forbidden | `However` 46/21 in sources, **0** in the corpus | amend §2c/§2d to license a claim beside its counter-consideration, narrowly |
| no plan-genre model | `REGISTER_EXEMPLAR.md` has no plan passage; 10 of 20 documents are plans | add a moves catalogue, seven patterns with verbatim examples |
| the given-new rule is unmet | 35 % chained against 57–60 % | exemplify §2d, which already states it, and check it |
| the surviving connective is capped | `therefore` capped at 1.2/1k, eight others unmentioned | drop the ceiling or pair it with the rest |
| sentences without events | `be` heads 33 % against 15–18 % | "the main verb names the event", with the mined examples |
| referents hidden behind possessives | `its` at 6.67/1k against 0.28 | "prefer the definite article or the noun itself" |
| **eight null results** | see §7 | **no rule written**; six plausible rules discarded |

## The rule about these rules

**Every measurement here is a diagnosis, never a target.** An author told to produce
`However` will produce `However`, and the metric will improve while the prose does not. This
repository has already watched that happen: when a one-sided sentence-length cap was added, the
next generation came back at a 17-word mean with 41 % of sentences under 15 words. The number
moved and the writing got worse.

So the acceptance test for the re-authored document is **discrimination, not counts**: can a
reader tell a corpus passage from a source passage? These features are re-run afterwards only
to see whether the shape moved, and they never reach the author as instructions.


## 13. The pilot, measured

TASK-009. `PCP-003` and `PCR-003` were re-authored on 2026-08-17 from the amended guide, the
moves catalogue and the discrepancy-bearing brief. This section measures the five shapes
before and after, against **all four** human sources rather than two.

"Before" is the text as it stood at commit `b0361f1`, kept in `pre-rewrite/` beside this
notebook so the comparison reproduces without a git checkout. Every rate is printed with its
denominator.


In [17]:
from check_style import HUMAN_SOURCES, measure, LIMITS

WORK = ROOT / '.claude/work/2026-08-16_01_register-from-four-sources'

# Four sources, at the page ranges check_style itself calibrates its thresholds on, so the
# reference columns here cannot drift from the gate's.
REF4 = {name: prose_from_extract(str(ROOT / 'refs/text' / f), lo, hi)
        for name, f, lo, hi in HUMAN_SOURCES}
PILOT = {
    'PCP-003 before': prose_from_qmd(str(WORK / 'pre-rewrite/PCP-003_bioreactor.qmd')),
    'PCP-003 after':  prose_from_qmd(str(ROOT / 'pc_package/PCP-003_bioreactor.qmd')),
    'PCR-003 before': prose_from_qmd(str(WORK / 'pre-rewrite/PCR-003_bioreactor.qmd')),
    'PCR-003 after':  prose_from_qmd(str(ROOT / 'pc_package/PCR-003_bioreactor.qmd')),
}
ALL8 = {**REF4, **PILOT}
S8 = {k: [s.replace('NUM', '12.3') for s in sentences(v)] for k, v in ALL8.items()}
for k, v in S8.items():
    print(f'{k:<17s} {len(v):5d} sentences, {len(ALL8[k].split()):6d} words')


PDA TR 60           820 sentences,  19864 words
A-Mab case study   1041 sentences,  27987 words
ISPE TT             669 sentences,  22216 words
ISPE PV             808 sentences,  24745 words
PCP-003 before      202 sentences,   4719 words
PCP-003 after       226 sentences,   5491 words
PCR-003 before      433 sentences,  10354 words
PCR-003 after       423 sentences,   9631 words


### Measure 1 — topic chaining

The largest single finding of section 8, and the one the amendment was least able to reach
directly: no rule was added, because a floor on chaining is met by typing a pronoun.


In [18]:
CHAIN = {}
for k, ss in S8.items():
    docs = [nlp(s) for s in ss[:600]]
    chained = pairs = 0
    for prev, cur in zip(docs, docs[1:]):
        subj = next((t for t in cur if t.dep_ in ('nsubj', 'nsubjpass')), None)
        if subj is None:
            continue
        pairs += 1
        prev_l = {t.lemma_.lower() for t in prev if t.pos_ in ('NOUN', 'PROPN', 'ADJ')}
        subj_l = {t.lemma_.lower() for t in subj.subtree if t.pos_ in ('NOUN', 'PROPN', 'ADJ')}
        chained += bool(subj_l & prev_l) or subj.pos_ == 'PRON'
    CHAIN[k] = (chained, pairs)
    print(f'{k:<17s} {100*chained/pairs:5.1f} %   ({chained}/{pairs} sentence pairs)')


PDA TR 60          59.4 %   (332/559 sentence pairs)


A-Mab case study   59.0 %   (315/534 sentence pairs)


ISPE TT            61.9 %   (348/562 sentence pairs)


ISPE PV            57.0 %   (321/563 sentence pairs)


PCP-003 before     31.0 %   (62/200 sentence pairs)


PCP-003 after      34.4 %   (77/224 sentence pairs)


PCR-003 before     35.1 %   (148/422 sentence pairs)


PCR-003 after      30.7 %   (127/414 sentence pairs)


### Measure 2 — the connective repertoire

Taken from `check_style.measure`, so these are the numbers the gate prints on every run.


In [19]:
MEAS8 = {k: measure(v)[0] for k, v in ALL8.items()}
for k, m in MEAS8.items():
    tot = sum(m['_connectives'].values())
    used = sum(1 for n in m['_connectives'].values() if n)
    print(f"{k:<17s} {1000*tot/m['_n_words']:5.2f} per 1k  {used}/9 distinct  "
          f"({tot} in {m['_n_words']} words)")


PDA TR 60          2.72 per 1k  9/9 distinct  (54 in 19856 words)
A-Mab case study   2.68 per 1k  7/9 distinct  (74 in 27649 words)
ISPE TT            2.24 per 1k  7/9 distinct  (42 in 18731 words)
ISPE PV            2.62 per 1k  6/9 distinct  (64 in 24425 words)
PCP-003 before     1.27 per 1k  3/9 distinct  (6 in 4718 words)
PCP-003 after      3.46 per 1k  6/9 distinct  (19 in 5489 words)
PCR-003 before     3.67 per 1k  3/9 distinct  (38 in 10346 words)
PCR-003 after      3.54 per 1k  6/9 distinct  (34 in 9614 words)


### Measures 3 and 5 — copula main verb, and the adjunct front field

A sentence is counted as *copula* when its ROOT lemma is `be`, and as having a *front field*
when any non-punctuation token precedes the subject phrase.


In [20]:
COPFRONT = {}
for k, ss in S8.items():
    cop = front = n = 0
    for s in ss[:450]:
        doc = nlp(s)
        root = next((t for t in doc if t.dep_ == 'ROOT'), None)
        subj = next((t for t in doc if t.dep_ in ('nsubj', 'nsubjpass')), None)
        if root is None or subj is None:
            continue
        n += 1
        cop += int(root.lemma_ == 'be')
        start = min(t.i for t in subj.subtree)
        front += bool([t for t in doc[:start] if not t.is_punct and not t.is_space])
    COPFRONT[k] = (cop, front, n)
    print(f'{k:<17s} copula {100*cop/n:5.1f} % ({cop}/{n})    '
          f'front field {100*front/n:5.1f} % ({front}/{n})')


PDA TR 60         copula  17.6 % (74/420)    front field  27.1 % (114/420)


A-Mab case study  copula  14.8 % (61/412)    front field  33.5 % (138/412)


ISPE TT           copula  22.4 % (95/424)    front field  35.6 % (151/424)


ISPE PV           copula  26.1 % (110/422)    front field  36.3 % (153/422)


PCP-003 before    copula  18.4 % (37/201)    front field  11.9 % (24/201)


PCP-003 after     copula  27.6 % (62/225)    front field  10.2 % (23/225)


PCR-003 before    copula  34.0 % (144/423)    front field  14.7 % (62/423)


PCR-003 after     copula  32.5 % (135/415)    front field   9.2 % (38/415)


### Measure 4 — possessives

The divergence found by ranking word frequencies rather than by reading.


In [21]:
POSS = {}
for k, txt in ALL8.items():
    w = len(txt.split())
    row = tuple(1000 * len(re.findall(p, txt, re.I)) / w
                for p in (r'\bits\b', r'\btheir\b', r'\bit\b'))
    POSS[k] = (row, w)
    print(f'{k:<17s} its {row[0]:5.2f}   their {row[1]:5.2f}   it {row[2]:5.2f}   ({w} words)')


PDA TR 60         its  0.40   their  0.96   it  3.12   (19864 words)
A-Mab case study  its  0.32   their  0.50   it  1.75   (27987 words)
ISPE TT           its  0.27   their  0.63   it  3.33   (22216 words)
ISPE PV           its  0.36   their  0.69   it  3.19   (24745 words)
PCP-003 before    its  5.72   their  3.18   it  7.42   (4719 words)
PCP-003 after     its  0.36   their  0.18   it 12.02   (5491 words)
PCR-003 before    its  6.66   their  4.15   it 10.62   (10354 words)
PCR-003 after     its  0.31   their  0.83   it  8.20   (9631 words)


### The register gate's own numbers, and the headroom question

TASK-002 raised `mean_len` to 30.5, `pct_over_40` to 21.5 and `pct_over_55` to 9.5 so that
ISPE PV would pass, whose extraction fuses list items into pseudo-sentences. If a re-authored
document lands near those ceilings rather than near the per-source columns, the widened band
is doing harm and that is a finding about the gate, not about the author.


In [22]:
GATE = ['mean_len', 'median_len', 'pct_over_40', 'pct_over_55', 'pct_under_15']
hdr = f"{'':<17s}" + ''.join(f'{g:>13s}' for g in GATE)
print(hdr)
print(f"{'band':<17s}" + ''.join(f'{str(LIMITS[g][0])+chr(45)+str(LIMITS[g][1]):>13s}' for g in GATE))
print('-' * len(hdr))
for k, m in MEAS8.items():
    print(f'{k:<17s}' + ''.join(f'{m[g]:>13.1f}' for g in GATE))


                      mean_len   median_len  pct_over_40  pct_over_55 pct_under_15
band                 20.0-30.5    18.0-26.5     3.0-21.5     None-9.5    15.0-32.0
----------------------------------------------------------------------------------
PDA TR 60                 24.2         21.0          9.8          2.9         20.5
A-Mab case study          26.6         23.0         13.4          5.2         19.5
ISPE TT                   28.0         24.0         14.8          5.8         16.3
ISPE PV                   30.2         26.0         20.8          9.0         16.2
PCP-003 before            23.4         22.0          7.9          0.0         18.3
PCP-003 after             24.3         24.0          6.6          0.4         20.4
PCR-003 before            23.9         24.0          5.8          0.0         22.4
PCR-003 after             22.7         22.0          4.5          0.2         22.7
